# GMM with Incomplete Data: Step-by-Step Experiment


## 1. Environment & Initialization

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../src')
from utils import generate_synthetic_data, mean_imputation
from gmm_missing import GMMMissing

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)

## 2. Synthetic Data Generation
We generate a 2D dataset with three clusters and deliberately introduce 30% Missing Completely At Random (MCAR) values to test the algorithm's imputation mechanism.

In [ ]:
X_true, X_incomplete, y_true, mask_missing = generate_synthetic_data(
    n_samples=200, n_features=2, centers=3, cluster_std=1.2, missing_ratio=0.3, random_state=42
)

# Trực quan hóa dữ liệu gốc vs khi bị xóa
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_true[:, 0], X_true[:, 1], c=y_true, cmap='viridis', s=30, alpha=0.7)
plt.title("Dữ liệu gốc (Ground Truth)")

plt.subplot(1, 2, 2)
complete_rows = ~mask_missing.any(axis=1)
plt.scatter(X_incomplete[complete_rows, 0], X_incomplete[complete_rows, 1], 
            c=y_true[complete_rows], cmap='viridis', s=30, alpha=0.7)
plt.title("Dữ liệu có thể quan sát (Đã lọc bỏ các sample chứa NaN)")
plt.show()

print(f"Tổng số mẫu: {len(X_incomplete)}")
print(f"Số lượng mẫu bị thiếu ít nhất 1 feature (NaN): {np.sum(mask_missing.any(axis=1))}")

## 3. Initialization
Standard GMM algorithms cannot handle `NaN` values directly. Therefore, following Step 1 of Algorithm 1:
1. Temporarily replace NaNs using column mean imputation.
2. Initialize parameters $\mu$ (mean), $\Sigma$ (covariance matrix), and $\alpha$ (mixing weights) using the K-Means algorithm.

In [ ]:
# Nội suy thô bằng Mean
X_initial = mean_imputation(X_incomplete)

gmm = GMMMissing(n_components=3,max_iter=1, tol=1e-4, random_state=42)
gmm._initialize_parameters(X_initial)

print("Trung bình các cụm (mu) khởi tạo từ K-Means:\n", gmm.mu_)

## 4. E-Step: Computing Posterior Probabilities (Expectation)
Calculate the posterior probability $\gamma_{ji}$ (responsibility), representing the probability that sample $x_j$ belongs to cluster $i$:

$$ \gamma_{ji} = \frac{\alpha_i p(x_j | \mu_i, \Sigma_i)}{\sum_{l=1}^k \alpha_l p(x_j | \mu_l, \Sigma_l)} $$

This step evaluates the "proximity" of each data point to every Gaussian component.

In [ ]:
log_likelihood, gamma = gmm._e_step(X_initial)
print("Xác suất hậu nghiệm gamma (5 mẫu đầu tiên):\n", gamma[:5])

## 5. M-Step: Updating Model Parameters
Given the responsibilities $\gamma_{ji}$, the algorithm updates the three GMM components (Equations 9, 10, 11):
- Updated mixing weights: $\alpha_i = \frac{\sum_{j=1}^n \gamma_{ji}}{n}$ (Implemented in `src/gmm_missing.py`, line 111).
- Updated means: $\mu_i = \frac{\sum_{j=1}^n \gamma_{ji} x_j}{\sum_{j=1}^n \gamma_{ji}}$ (Equation 9, line 116).
- Updated covariances: $\Sigma_i$ (Equation 10, line 120).

In [ ]:
gmm._m_step_params(X_initial, gamma)
print("Tâm cụm sau 1 vòng M-Step:\n", gmm.mu_)

## 6. M-Step: Dynamic Imputation of Missing Data
**The core contribution:** Instead of relying on static initial imputations, the model treats the missing data $\mathbf{x}_{jm}$ as latent variables to be optimized.

### Equation 13: Data Partitioning
For sample $j$, the features are partitioned into observed ($o$) and missing ($m$) components.
$$ x_j = \begin{bmatrix} x_{jo} \\ x_{jm} \end{bmatrix}, \quad \mu_i = \begin{bmatrix} \mu_{io} \\ \mu_{im} \end{bmatrix} $$

The mathematical focus lies in partitioning the **Precision Matrix** $\Sigma^{-1}$, denoted as $\Lambda$ for brevity:
$$ \Sigma_i^{-1} = \Lambda_i = \begin{bmatrix} \Lambda_{ioo} & \Lambda_{iom} \\ \Lambda_{imo} & \Lambda_{imm} \end{bmatrix} $$

> ⚠️ **Implementation Note:** In Equation 13, the notation $\Sigma_{imm}^{-1}$ refers to the $(m,m)$ block of the **inverse matrix** (i.e., $\Lambda_{imm}$), *not* the inverse of the $(m,m)$ block of the original covariance matrix. Our implementation in `_m_step_data()` (line 137) correctly computes `np.linalg.inv(covariances)` before extracting the `L_mm` and `L_mo` blocks via masking.

### Equation 14: Analytical Computation of $\mathbf{x}_{jm}$
By setting the partial derivative of the total log-likelihood with respect to $x_{jm}$ to zero, we obtain an explicit analytical solution:
$$ X_m = \left( \sum_{i=1}^k \gamma_{ji} \Lambda_{imm} \right)^{-1} \sum_{i=1}^k \gamma_{ji} \left( \Lambda_{imm} \mu_{im} - \Lambda_{imo}(x_{jo} - \mu_{io}) \right) $$

*(In the code, we use $\gamma_{ji}$ instead of $\alpha_i P_i$; since the derivative is set to zero, constant scaling factors do not affect the result. Furthermore, $\gamma_{ji}$ helps mitigate numerical underflow). The logic is demonstrated in the simulation block below:*

In [ ]:
# --- TRỰC QUAN HÓA LOGIC PHƯƠNG TRÌNH 14 TRONG gmm_missing.py ---
# Lấy Sample số 0 làm ví dụ
sample_index = 0

# Tách mask cho sample 0
m_part = mask_missing[sample_index]     # Các chiều bị missing (True)
o_part = ~mask_missing[sample_index]    # Các chiều có dữ liệu (False)

if np.any(m_part):
    print(f"Sample {sample_index} gốc có NaNs: {X_incomplete[sample_index]}")
    x_j_o = X_initial[sample_index, o_part]
    
    # Tổng tích lũy ma trận theo k (k=3 cụm)
    sum_L_mm = np.zeros((np.sum(m_part), np.sum(m_part)))
    sum_term2 = np.zeros(np.sum(m_part))
    
    for k in range(gmm.n_components):
        gamma_jk = gamma[sample_index, k]
        Lambda_k = np.linalg.inv(gmm.covariances_[k])
        
        # Trích xuất L_mm và L_mo (Phương trình 13)
        L_mm = Lambda_k[np.ix_(m_part, m_part)]
        L_mo = Lambda_k[np.ix_(m_part, o_part)]
        
        mu_m = gmm.mu_[k, m_part]
        mu_o = gmm.mu_[k, o_part]
        
        # Phương trình 14 tích lũy
        sum_L_mm += gamma_jk * L_mm
        term2 = L_mm @ mu_m - L_mo @ (x_j_o - mu_o)
        sum_term2 += gamma_jk * term2
        
    # Cập nhật kết quả cuối cùng
    x_j_m_updated = np.linalg.inv(sum_L_mm) @ sum_term2
    
    print(f"Dữ liệu nội suy thô (Mean): {X_initial[sample_index]}")
    print(f"Tọa độ bị thiếu NAY ĐÃ ĐƯỢC KÉO về mốc mới: {x_j_m_updated}")

## 7. Iterative Convergence (Full EM Cycle)
The logic (E-Step -> Parameter M-Step -> Data M-Step) iterates until the log-likelihood objective converges. Through this interdependent optimization, missing values progressively migrate toward their most probable locations relative to the cluster components.

In [ ]:
# Chạy luồng fit hoàn chỉnh từ module
gmm_full = GMMMissing(n_components=3, max_iter=200, random_state=42)
gmm_full.fit(X_incomplete)

print(f"Thuật toán hội tụ thành công sau {gmm_full.n_iter_} vòng lặp.")

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.scatter(X_true[:, 0], X_true[:, 1], c=y_true, cmap='viridis', s=30, alpha=0.5)
plt.title("Dữ liệu gốc hoàn chỉnh")

plt.subplot(1, 2, 2)
y_pred = gmm_full.predict(gmm_full.X_final_)
plt.scatter(gmm_full.X_final_[:, 0], gmm_full.X_final_[:, 1], c=y_pred, cmap='viridis', s=30, alpha=0.5)

# Đánh dấu viền đỏ cho các tọa độ từng bị mất (nay đã nội suy)
missing_rows_idx = np.where(mask_missing.any(axis=1))[0]
plt.scatter(gmm_full.X_final_[missing_rows_idx, 0], gmm_full.X_final_[missing_rows_idx, 1], 
            facecolors='none', edgecolors='red', s=60, label='Được nội suy động')

plt.title("GMM Clustering kết hợp Dynamic Imputation (Viền đỏ)")
plt.legend()
plt.show()